# Multiple band likelihood setup

In [1]:
%reset -f
%run setup_notebook.py

from importlib import reload
import matplotlib.pyplot as plt
import numpy as np
from like3 import pixel_table as _pt; reload(_pt)
PixelTable = _pt.PixelTable
from like3 import sourcelist as _sl; reload(_sl)
SourceModel = _sl.SourceModel
pt = PixelTable('files/kerr/toby_v4.fits')
sm = SourceModel.from_fermi_catalog('Geminga', version='v36')
sm

Loaded pixel table from "files/kerr/toby_v4.fits":
            42 bands Band(0, 4): PSF0@1.33 GeV nside 64 occ 1.001 ... Band(3, 11): PSF3@74.99 GeV nside 2048 occ 0.000
            193,317,496 photons
            8,240,918 pixels, order ring
            
Loaded Fermi 4FGL gll_psc_v36.fit: 7221 entries


SourceModel: @Geminga (v36) 1 sources with 2 free parameters

In [4]:
from like3 import main as _main; reload(_main)
MultiBandLikelihood = _main.MultiBandLikelihood
mbl = MultiBandLikelihood(pt, sm)
mbl.select(energy = 1333.0)

[(0, 4), (1, 4), (2, 4), (3, 4)]

In [5]:
mbl.loglike_grad()

(1542318.571102496, array([-3956.52088338,  -459.56197655]))

In [9]:
display(f""" ## Numerical gradient check for BandLikelihood""")

def gradient_check(bl,  eps=1e-3):
    """Compare analytic gradient from grad_fn to numerical gradient of loglike_fn at pars."""

    def numerical_gradient(loglike_fn, pars, eps=1e-3):
        """Compute numerical gradient of loglike_fn at pars using central differences."""
        grad = np.zeros_like(pars, dtype=float)
        for i in range(len(pars)):
            p_hi = np.array(pars, dtype=float)
            p_lo = np.array(pars, dtype=float)
            p_hi[i] += eps
            p_lo[i] -= eps
            f_hi = loglike_fn(p_hi)
            f_lo = loglike_fn(p_lo)
            grad[i] = (f_hi - f_lo) / (2 * eps)
        return grad
    pars0 = bl.parameters# if hasattr(bl.source_model, 'parameters') else np.concatenate([src.model.parameters[src.model.free] for src in bl.source_model])
    ll, analytic_grad = bl.loglike_grad(pars0)
    num_grad = numerical_gradient(lambda p: bl.loglike(p), pars0)
    bl.parameters = pars0

    num_grad = numerical_gradient(bl.loglike, pars0, eps=1e-3)
    print(f'Analytic gradient: {analytic_grad.round().astype(int)}')
    print(f'Numerical gradient: {num_grad.round().astype(int)}')
    print(f'Difference: {(analytic_grad - num_grad).round(1)}')

gradient_check(mbl)


' ## Numerical gradient check for BandLikelihood'

Analytic gradient: [-3957  -460]
Numerical gradient: [-3957  -460]
Difference: [0.4 0. ]
